In [1]:
import numpy as np
import time
from scipy.io import savemat

# -----------------------------
# Parameters
# -----------------------------
nsx = 32
nsy = 32
nbz = 16
nx = nsx
ny = nsy

len_ = 50.0
z_obs = -0.1
UGC = 6.67e-11  # Gravitational constant

# -----------------------------
# Grid definition
# -----------------------------
Xrange = [len_ / 2, nsx * len_ - len_ / 2]
Yrange = [len_ / 2, nsy * len_ - len_ / 2]
Xsrange = [0, nsx * len_]
Ysrange = [0, nsy * len_]
Zsrange = [0, nbz * len_]

Xobserved = np.linspace(Xrange[0], Xrange[1], nx)
Yobserved = np.linspace(Yrange[0], Yrange[1], ny)

position_nsx = np.linspace(Xsrange[0], Xsrange[1], nsx + 1)
position_nsy = np.linspace(Ysrange[0], Ysrange[1], nsy + 1)
position_nbz = np.linspace(Zsrange[0], Zsrange[1], nbz + 1)

# -----------------------------
# Initialize G matrix
# -----------------------------
Gz = np.zeros((nx * ny, nsx * nsy * nbz))

# -----------------------------
# Compute Gz
# -----------------------------
start = time.time()

for i1 in range(nx):
    for i2 in range(ny):

        x = Xobserved[i1]
        y = Yobserved[i2]

        for j1 in range(nsx):
#print(f"Computation finished in {end - start:.2f} seconds")
            for j2 in range(nsy):
                for j3 in range(nbz):

                    x1 = position_nsx[j1]
                    x2 = position_nsx[j1 + 1]

                    y1 = position_nsy[j2]
                    y2 = position_nsy[j2 + 1]

                    z1 = position_nbz[j3]
                    z2 = position_nbz[j3 + 1]

                    a1, a2 = x - x1, x - x2
                    b1, b2 = y - y1, y - y2
                    c1, c2 = z_obs - z1, z_obs - z2

                    # Distances
                    r111 = np.sqrt(a1**2 + b1**2 + c1**2)
                    r112 = np.sqrt(a1**2 + b1**2 + c2**2)
                    r121 = np.sqrt(a1**2 + b2**2 + c1**2)
                    r211 = np.sqrt(a2**2 + b1**2 + c1**2)
                    r122 = np.sqrt(a1**2 + b2**2 + c2**2)
                    r212 = np.sqrt(a2**2 + b1**2 + c2**2)
                    r221 = np.sqrt(a2**2 + b2**2 + c1**2)
                    r222 = np.sqrt(a2**2 + b2**2 + c2**2)

                    # Terms
                    term111 = -1 * (a1 * np.log(b1 + r111) + b1 * np.log(a1 + r111) - c1 * np.arctan((a1 * b1) / (c1 * r111)))
                    term112 =  1 * (a1 * np.log(b1 + r112) + b1 * np.log(a1 + r112) - c2 * np.arctan((a1 * b1) / (c2 * r112)))
                    term121 =  1 * (a1 * np.log(b2 + r121) + b2 * np.log(a1 + r121) - c1 * np.arctan((a1 * b2) / (c1 * r121)))
                    term211 =  1 * (a2 * np.log(b1 + r211) + b1 * np.log(a2 + r211) - c1 * np.arctan((a2 * b1) / (c1 * r211)))
                    term122 = -1 * (a1 * np.log(b2 + r122) + b2 * np.log(a1 + r122) - c2 * np.arctan((a1 * b2) / (c2 * r122)))
                    term212 = -1 * (a2 * np.log(b1 + r212) + b1 * np.log(a2 + r212) - c2 * np.arctan((a2 * b1) / (c2 * r212)))
                    term221 = -1 * (a2 * np.log(b2 + r221) + b2 * np.log(a2 + r221) - c1 * np.arctan((a2 * b2) / (c1 * r221)))
                    term222 =  1 * (a2 * np.log(b2 + r222) + b2 * np.log(a2 + r222) - c2 * np.arctan((a2 * b2) / (c2 * r222)))

                    index1 = i1 + nx * i2
                    index2 = j1 + nsx * j2 + nsx * nsy * j3

                    Gz[index1, index2] = -UGC * (
                        term111 + term112 + term121 + term211 +term122 + term212 + term221 + term222) * 1e5  # Convert to mGal

end = time.time()


print(f"Computation finished in {end - start:.2f} seconds")

# -----------------------------
# Save G matrix to .mat file
# -----------------------------
savemat('G_matrix.mat', {'Gz': Gz})
print("✅ G matrix saved as 'Gz_matrix.mat'")


Computation finished in 615.12 seconds
✅ G matrix saved as 'Gz_matrix.mat'


In [3]:
print(G.shape)

(1024, 16384)
